# tranSPORTmer_D · 데드볼 전용

- **비교 목적** · 공동 학습의 공 복원 손실 가중치만 0으로 둔다. 공 토큰·CLS·선수별 attention과 초기화는 유지한다.
- **설정** · 128차원, 16헤드, 9.6초 창, AdamW, 최대 60 epoch. 학습 손실은 4 × 데드볼 BCE.
- **입력과 검증** · 공동 학습과 같은 DFL 7경기 Parquet, 입력 해시, 선수 노이즈, 내부 검증 창, 조기 종료, 6경기 학습·1경기 평가를 사용한다.
- **실행** · 같은 폴더의 `dfl_transportmer_joint.ipynb`와 `dfl_transportmer.ipynb`가 필요하다. 같은 Python 환경에서 셀을 순서대로 실행한다.
- **출력** · `model/영상추출/추출csv/dead_ball/transportmer_d_20261112_01/`. 데드볼 프레임 F1과 인플레이 구간 F1을 공동 학습과 비교한다.

In [ ]:
import json
from pathlib import Path

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/train_dfl.py").is_file())
joint = root / "model/영상추출/모델/dead_ball/dfl_transportmer_joint.ipynb"
joint_cells = {cell["id"]: cell for cell in json.loads(joint.read_text())["cells"]}
for index in (2, 4, 6):
    cell = joint_cells[f"dfl-transportmer-joint-{index}"]
    exec(compile("".join(cell["source"]), f"{joint}:{index}", "exec"))

JOINT_RESULTS = OUT
RUN_ID = "transportmer_d_20261112_01"
OUT = ROOT / "model/영상추출/추출csv/dead_ball" / RUN_ID
NOTEBOOK = folder / "tranSPORTmer_D.ipynb"
REFERENCE = json.loads((JOINT_RESULTS / "run_manifest.json").read_text())
assert digest(joint) == REFERENCE["fingerprint"]["notebook_sha256"]
assert digest(original) == REFERENCE["fingerprint"]["base_notebook_sha256"]
assert AMP == REFERENCE["fingerprint"]["amp"]
config = dict(width=WIDTH, heads=HEADS, max_epochs=EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA,
              min_epochs=MIN_EPOCHS, inner_share=INNER_SHARE, state_weight=STATE_WEIGHT,
              batch=BATCH, lr=LR, step=STEP, window=WINDOW, seed=SEED)
assert config == {key: REFERENCE["config"][key] for key in config}
assert (WIDTH, HEADS, EPOCHS) == (128, 16, 60)

## 데드볼 손실과 평가

최고 내부 점수의 모델 가중치를 메모리에 보관하고 해당 모델로 바깥 경기를 평가한다. 가려진 공 토큰은 공동 학습과 같은 입력 조건을 유지하는 데 사용한다.

In [ ]:
def train_state_batch(model, opt, data, indices):
    x, valid, label = move(data, indices, include_target=True)
    opt.zero_grad(set_to_none=True)
    with amp():
        logits, _ = model(x, valid)
    bce = (F.binary_cross_entropy_with_logits(logits.float(), label.clamp_min(0), reduction="none") * valid).sum()
    bce = bce / valid.sum()
    (STATE_WEIGHT * bce).backward()
    nn.utils.clip_grad_norm_(model.parameters(), 5.0, foreach=False)
    opt.step()
    return bce.detach()


def verify_state_only():
    parameters = check_joint()
    assert parameters == REFERENCE["config"]["model_parameters"]
    model, opt = new_joint()
    x = np.zeros((2, WINDOW, SLOTS, 4), dtype=np.float32)
    x[:, :, :2, :2] = np.random.default_rng(SEED).normal(size=(2, WINDOW, 2, 2))
    x[:, :, :2, 3] = 1
    data = dict(x=x, valid=np.ones((2, WINDOW), dtype=bool),
                target=np.tile(np.arange(WINDOW) % 2, (2, 1)))
    loss = train_state_batch(model, opt, data, slice(None))
    assert torch.isfinite(loss).item()
    assert all(parameter.grad is None for parameter in model.ball_head.parameters())
    assert any(parameter.grad is not None and parameter.grad.abs().sum().item() > 0
               for parameter in model.classifier.parameters())
    return parameters


def fit_state(held, matches, datasets, splits):
    model, opt = new_joint()
    train = {key: np.concatenate([datasets[m][key][splits[m][0]] for m in matches])
             for key in ("x", "valid", "target")}
    best_score, best_epoch, best, stale = -np.inf, 0, None, 0
    started = perf_counter()
    for epoch in range(1, EPOCHS + 1):
        tick = perf_counter()
        model.train()
        order = np.random.default_rng(SEED + epoch).permutation(len(train["x"]))
        losses = [train_state_batch(model, opt, train, order[i:i+BATCH])
                  for i in range(0, len(order), BATCH)]
        bce = float(torch.stack(losses).mean().item())
        assert np.isfinite(bce)
        validation = inner_f1(model, datasets, splits, matches) if epoch >= MIN_EPOCHS else None
        if validation is not None:
            if validation > best_score + MIN_DELTA:
                best_score, best_epoch, best, stale = validation, epoch, cpu_weights(model), 0
            else:
                stale += 1
        state = "대기" if validation is None else f"{validation:.4f}"
        print(f"  {held} epoch {epoch:02}/{EPOCHS} | {perf_counter()-tick:.1f}s | BCE {bce:.4f} | "
              f"내부 구간 F1 {state} | best {best_epoch} | patience {stale}/{PATIENCE}", flush=True)
        if stale >= PATIENCE:
            break
    assert best is not None and best_epoch >= MIN_EPOCHS
    model.load_state_dict(best)
    return model.eval(), epoch, best_epoch, best_score, perf_counter() - started


class StateOutput(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, players, valid):
        with amp():
            state, _ = self.model(players, valid)
        return state.float()

## 실행 및 산출물

`per_match.csv`, `model_summary.csv`, `comparison.csv`에 두 지표를 저장한다. 경기별 `features.csv`와 `frame_status.csv`의 컬럼은 공동 학습과 같다.

In [ ]:
def run_state_only(parameters):
    OUT.mkdir(parents=True, exist_ok=True)
    started = perf_counter()
    datasets, evaluations, splits, inputs = {}, {}, {}, {}
    for match in MATCHES:
        for path in map(Path, dfl_load.cache_paths(match)):
            inputs[str(path.relative_to(ROOT))] = digest(path)
        datasets[match], evaluations[match] = prepare_joint(match)
        splits[match] = internal_split(datasets[match])
        for key in ("ball", "ball_target"):
            datasets[match].pop(key)
        print(f"{match} 준비 | 학습 창 {len(splits[match][0])} | 내부검증 창 "
              f"{sum(map(len, splits[match][1]))} | 바깥 검증 프레임 {len(evaluations[match]['label']):,}", flush=True)
    assert inputs == REFERENCE["fingerprint"]["input_sha256"]
    preparation_s = perf_counter() - started
    results = []

    for held in MATCHES:
        started = perf_counter()
        matches = [match for match in MATCHES if match != held]
        assert len(matches) == 6
        print(f"{held} 바깥 평가 | 학습/내부검증 {', '.join(matches)}", flush=True)
        model, stopped, best, best_score, fit_s = fit_state(held, matches, datasets, splits)
        evaluation = evaluations[held]
        probability, sampled = predict(StateOutput(model), evaluation)
        scores, viterbi, forced = score_joint(evaluation, probability)
        files = write_frames(held, evaluation, probability, sampled, viterbi, forced)
        row = dict(match=held, train_matches=matches, best_epoch=best, stopped_epoch=stopped,
                   best_inner_f1=best_score, frames=len(probability), training_s=fit_s,
                   fold_s=perf_counter()-started, scores=scores,
                   file_sha256={str(path.relative_to(OUT)): digest(path) for path in files})
        results.append(row)
        print(f"{held} 완료 | epoch {best}/{stopped} | 보정 후 프레임 F1 {scores[-1]['dead_f1']:.4f} "
              f"| 구간 F1 {scores[-1]['alive_spell_f1']:.4f} | 전체 {row['fold_s']:.1f}s", flush=True)

    detail = pd.DataFrame([
        dict(match=fold["match"], best_epoch=fold["best_epoch"], stopped_epoch=fold["stopped_epoch"],
             training_s=fold["training_s"], fold_s=fold["fold_s"], **row)
        for fold in results for row in fold["scores"]
    ])
    assert len(detail) == 21 and not detail.duplicated(["match", "stage"]).any()
    assert set(detail.match) == set(MATCHES)
    assert detail.dead_f1.between(0, 1).all() and detail.alive_spell_f1.between(0, 1).all()
    summary = (detail.groupby("stage", sort=False)
               .agg(dead_frame_f1=("dead_f1", "mean"), dead_recall=("dead_recall", "mean"),
                    dead_precision=("dead_precision", "mean"), frame_acc=("frame_acc", "mean"),
                    alive_spell_f1=("alive_spell_f1", "mean"), training_s=("training_s", "sum"),
                    fold_s=("fold_s", "sum"), min_best_epoch=("best_epoch", "min"),
                    max_best_epoch=("best_epoch", "max")).reset_index())
    shared = ["stage", "dead_frame_f1", "alive_spell_f1"]
    comparison = pd.concat([
        summary[shared].assign(model="tranSPORTmer_D"),
        pd.read_csv(JOINT_RESULTS/"model_summary.csv")[shared].assign(model="공동 학습"),
        pd.read_csv(OUT.parent/"dfl_lomo_20261111_01/model_summary.csv")
            .query("model == 'LightGBM'")[["model", *shared]],
    ], ignore_index=True)
    detail.to_csv(OUT/"per_match.csv", index=False)
    summary.to_csv(OUT/"model_summary.csv", index=False)
    comparison.to_csv(OUT/"comparison.csv", index=False)
    names = ("per_match.csv", "model_summary.csv", "comparison.csv")
    save_json(dict(run_id=RUN_ID, config=dict(config, ball_loss_weight=0, model_parameters=parameters),
                   notebook_sha256=digest(NOTEBOOK), joint_notebook_sha256=digest(joint),
                   base_notebook_sha256=digest(original), input_sha256=inputs,
                   device=str(DEVICE), amp=AMP, preparation_s=preparation_s, matches=results,
                   versions={name: version(name) for name in ("torch", "pandas", "numpy", "pyarrow")},
                   code_commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip(),
                   report_sha256={name:digest(OUT/name) for name in names}), OUT/"run_manifest.json")
    save_json(dict(passed=True, held_matches=7, score_rows=len(detail),
                   frames=sum(fold["frames"] for fold in results), smoke_test=True), OUT/"validation.json")
    print(comparison.round(4).to_string(index=False), flush=True)
    return summary


parameters = verify_state_only()
print(f"device={DEVICE} | width={WIDTH} | heads={HEADS} | params={parameters:,} | ball-loss weight 0", flush=True)
if os.environ.get("MTAH_TRANSPORTMER_MODE") != "check":
    summary = run_state_only(parameters)